human lung cancer

导入相关包

In [29]:
import scanpy as sc
import pandas as pd
import numpy as np
from sklearn.metrics import (homogeneity_score,v_measure_score,adjusted_mutual_info_score,normalized_mutual_info_score,adjusted_rand_score,fowlkes_mallows_score)


In [30]:
repeat_times = 2
simple_path = '/home/cavin/jt/benchmark/data/hbc/s1_filtered_cells.h5ad'
celltype_col = 'cell_type'
cell_emb_col = 'X_Nicheformer'
save_path = f"/home/cavin/jt/benchmark/experiments/embedings/batch_integrate/nicheformer_human_breast_cancer_emb.parquet"


读取嵌入

In [31]:
loaded_emb_df = pd.read_parquet(save_path)
adata = sc.read_h5ad(simple_path)
aligned_emb_df = loaded_emb_df.reindex(adata.obs_names)
adata

AnnData object with n_obs × n_vars = 281780 × 313
    obs: 'x_centroid', 'y_centroid', 'transcript_counts', 'control_probe_counts', 'control_codeword_counts', 'total_counts', 'cell_area', 'nucleus_area', 'batch', 'cell_type', 'n_genes'
    var: 'ensemble_id', 'type'
    obsm: 'spatial'

In [32]:
adata.obsm[cell_emb_col] = aligned_emb_df.to_numpy(dtype=np.float32)
adata

AnnData object with n_obs × n_vars = 281780 × 313
    obs: 'x_centroid', 'y_centroid', 'transcript_counts', 'control_probe_counts', 'control_codeword_counts', 'total_counts', 'cell_area', 'nucleus_area', 'batch', 'cell_type', 'n_genes'
    var: 'ensemble_id', 'type'
    obsm: 'spatial', 'X_Nicheformer'

In [33]:
res_list = [0.1,0.2,0.3,0.4,0.5,0.6,0.7,0.8,0.9,1.0]
random_seed=2026 + repeat_times * 200
key_added='leiden'
sc.pp.neighbors(adata, use_rep=cell_emb_col,random_state=random_seed)
adata


AnnData object with n_obs × n_vars = 281780 × 313
    obs: 'x_centroid', 'y_centroid', 'transcript_counts', 'control_probe_counts', 'control_codeword_counts', 'total_counts', 'cell_area', 'nucleus_area', 'batch', 'cell_type', 'n_genes'
    var: 'ensemble_id', 'type'
    uns: 'neighbors'
    obsm: 'spatial', 'X_Nicheformer'
    obsp: 'distances', 'connectivities'

In [34]:
def main(adata,random_seed,res_list,key_added,celltype_col):
    max_value = 0
    metrics = {"method": "Nicheformer", "ARI": 0, "NMI": 0, "AMI": 0, "HS": 0, "VM": 0, "FMI": 0, "mean value": 0}
    save_label_df = None
    for used_res in res_list:
        sc.tl.leiden(adata, random_state=random_seed, resolution=used_res,key_added=key_added)
        true_labels = np.array(adata.obs[celltype_col])
        cluster_labels = np.array(adata.obs[key_added])
        FMI = fowlkes_mallows_score(true_labels, cluster_labels)
        homogeneity = homogeneity_score(true_labels, cluster_labels)
        v_measure = v_measure_score(true_labels, cluster_labels)
        ami = adjusted_mutual_info_score(true_labels, cluster_labels)
        nmi = normalized_mutual_info_score(true_labels, cluster_labels)
        ari = adjusted_rand_score(true_labels, cluster_labels)
        mean_value = np.mean([FMI, homogeneity, v_measure, ami, nmi, ari])

        n_cluster = len(adata.obs[key_added].unique())
        n_celltype = len(adata.obs[celltype_col].unique())
        if mean_value > max_value:
            save_label_df = adata.obs[key_added]
            metrics["ARI"] = ari
            metrics["NMI"] = nmi
            metrics["AMI"] = ami
            metrics["HS"] = homogeneity
            metrics["VM"] = v_measure
            metrics["FMI"] = FMI
            metrics["mean value"] = mean_value
            max_value = mean_value

        print(f'resolution = {used_res} | n_celltype = {n_celltype} | n_cluster = {n_cluster} | mean_value = {round(mean_value,3)} | ARI = {round(ari,3)} |  NMI = {round(nmi,3)} | AMI = {round(ami,3)} | HS = {round(homogeneity,3)} | VM = {round(v_measure,3)} | FMI = {round(FMI,3)} ')

    return metrics, save_label_df

In [35]:
metrics, save_label_df = main(adata,random_seed,res_list,key_added,celltype_col)

resolution = 0.1 | n_celltype = 20 | n_cluster = 5 | mean_value = 0.306 | ARI = 0.217 |  NMI = 0.317 | AMI = 0.317 | HS = 0.228 | VM = 0.317 | FMI = 0.437 
resolution = 0.2 | n_celltype = 20 | n_cluster = 10 | mean_value = 0.327 | ARI = 0.205 |  NMI = 0.366 | AMI = 0.366 | HS = 0.34 | VM = 0.366 | FMI = 0.321 
resolution = 0.3 | n_celltype = 20 | n_cluster = 11 | mean_value = 0.347 | ARI = 0.22 |  NMI = 0.388 | AMI = 0.388 | HS = 0.369 | VM = 0.388 | FMI = 0.329 
resolution = 0.4 | n_celltype = 20 | n_cluster = 15 | mean_value = 0.379 | ARI = 0.239 |  NMI = 0.423 | AMI = 0.423 | HS = 0.432 | VM = 0.423 | FMI = 0.334 
resolution = 0.5 | n_celltype = 20 | n_cluster = 15 | mean_value = 0.377 | ARI = 0.237 |  NMI = 0.42 | AMI = 0.42 | HS = 0.432 | VM = 0.42 | FMI = 0.331 
resolution = 0.6 | n_celltype = 20 | n_cluster = 17 | mean_value = 0.376 | ARI = 0.233 |  NMI = 0.419 | AMI = 0.419 | HS = 0.442 | VM = 0.419 | FMI = 0.325 
resolution = 0.7 | n_celltype = 20 | n_cluster = 18 | mean_value

In [36]:
metrics

{'method': 'Nicheformer',
 'ARI': 0.23916373699995103,
 'NMI': 0.42273811134057776,
 'AMI': 0.4226234812248149,
 'HS': 0.4324824406511914,
 'VM': 0.4227381113405777,
 'FMI': 0.333646283702469,
 'mean value': 0.37889869420993033}

In [37]:
save_label_df

s1r1_1          3
s1r1_2          3
s1r1_5          3
s1r1_8          3
s1r1_9          3
               ..
s1r2_118748     2
s1r2_118749    10
s1r2_118750    11
s1r2_118751    11
s1r2_118752     2
Name: leiden, Length: 281780, dtype: category
Categories (15, object): ['0', '1', '2', '3', ..., '11', '12', '13', '14']

In [38]:
save_label_df_path = f"/home/cavin/jt/benchmark/experiments/results/labels_df/breast_cancer/nicheformer_human_breast_cancer_labels_repeat_{repeat_times}.csv"

In [39]:
save_label_df.to_csv(save_label_df_path)

In [40]:
metrics_df = pd.DataFrame.from_dict(metrics, orient='index').T

In [41]:
metrics_df_save_path = f"/home/cavin/jt/benchmark/experiments/results/cluster_metrics/breast_cancer/human_breast_cancer_metrics_repeat_{repeat_times}.csv"

In [42]:
metrics_df.to_csv(metrics_df_save_path, index=False,mode="a", header=not pd.io.common.file_exists(metrics_df_save_path))